In [2]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.3 s3fs==2026.3.0 -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Import des packages nécessaires

In [3]:
import pandas as pd
import io
from openpyxl import load_workbook
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.chart import BarChart, PieChart, Reference
from openpyxl.chart.series import SeriesLabel
from openpyxl.utils import get_column_letter

# Import des données
Les données sont disponibles sur le site [DATA AMELI](https://data.ameli.fr/pages/home/).




J’ai choisi de travailler avec deux jeux de données totalement distincts :
**Une vue régionale et départementale**, décrivant les patients selon la pathologie, le traitement chronique ou l’épisode de soins, le sexe, la classe d’âge, la région et le département.  
  - Source : [Effectifs – Patients par pathologie](https://data.ameli.fr/explore/dataset/effectifs/information/)

**Une vue nationale**, portant sur les dépenses remboursées par pathologie et par patient en moyenne.  
  - Source : [Dépenses remboursées par pathologie](https://data.ameli.fr/explore/dataset/depenses/information/)

Comme le fichier **effectifs** est particulièrement volumineux, j’ai opté pour une **lecture en chunks**, ce qui permet de filtrer les données au fur et à mesure du chargement et d’éviter une surcharge mémoire.

J’ai ensuite enrichi ces données en les fusionnant avec un second fichier **region_dept** contenant les libellés des régions et des départements, afin d’obtenir un rendu plus harmonieux et plus lisible dans les visualisations.

Pour garantir un chargement fiable et éviter les problèmes liés aux chemins locaux, j’ai préféré déposer mes fichiers CSV sur Onyxia et les récupérer directement via leur URL MinIO. Cette approche assure un accès stable et reproductible aux données, quel que soit l’environnement d’exécution.

### Effectif de patients par pathologie, sexe, classe d'âge et territoire (département, région)

In [4]:
chunks = []

for chunk in pd.read_csv('https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/effectifs.csv', sep=';', chunksize=100_000,low_memory=False):
    filtered = chunk[(chunk['annee'] >= 2022) & (chunk['dept'] != '999')]
    chunks.append(filtered)

df_effectifs = pd.concat(chunks, ignore_index=True)

#Le datasaet qui va servir de jointure pour récuperer les libelles des départements et région
df_regions_dept = pd.read_csv(
    "https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/LibelleRegionDept.csv",
    sep=";",
    encoding="latin1",
    usecols=["departement", "libelle_departement", "libelle_region"]
)

# Harmonisation des colonnes , les 1 devient des 01 
df_regions_dept["departement"] = df_regions_dept["departement"].astype(str).str.zfill(2)
df_effectifs["dept"] = df_effectifs["dept"].astype(str).str.zfill(2)

# Jointure des 2 :
df_fusion = pd.merge(
    df_effectifs, 
    df_regions_dept, 
    left_on="dept", 
    right_on="departement", 
    how="inner"
)



Ici, je n’utilise pas `.copy()` car mon objectif n’est pas de dupliquer le DataFrame, mais simplement de le renommer.  
En faisant :

In [5]:
df_effectifs = df_fusion
del df_fusion # Et par la suite je supprime le dataFrame pour libérer de l'espace

## Nettoyage du dataset : df_effectifs

### Vérification et traitement des doublons

J’affiche ici les lignes du DataFrame qui apparaissent en doublon afin d’identifier
les répétitions éventuelles dans les données. L’option `keep=False` permet de visualiser
toutes les occurrences d’un doublon, ce qui facilite le diagnostic avant nettoyage.

Le fichier des correspondances *région–département* contient plusieurs doublons,ce qui génère des lignes dupliquées après la jointure.
Pour éviter ces répétitions dans le DataFrame final, j’ai décidé de supprimer les doublons avant de poursuivre l’analyse.


In [6]:
df_effectifs[df_effectifs.duplicated(keep=False)]


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,libelle_region,departement,libelle_departement
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
1,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
2,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
3,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
4,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4878295,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
4878296,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
4878297,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
4878298,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques


#### Nombre de doublons par colonne

In [7]:
df_effectifs.apply(lambda col: col.duplicated().sum())


annee                  4878298
patho_niv1             4878282
patho_niv2             4878251
patho_niv3             4878238
top                    4878221
cla_age_5              4878279
sexe                   4878297
region                 4878282
dept                   4878199
Ntop                   4869980
Npop                   4872749
prev                   4834295
Niveau prioritaire     4878294
libelle_classe_age     4878279
libelle_sexe           4878297
tri                    4878221
libelle_region         4878282
departement            4878199
libelle_departement    4878199
dtype: int64

### Poids du dataset avant et après suppression des doublons

Avant de nettoyer les données, j’affiche le poids mémoire du DataFrame afin d’évaluer l’impact des doublons sur la taille totale du fichier.  
Après suppression des lignes dupliquées, j’affiche à nouveau le poids du dataset pour mesurer le gain en mémoire et vérifier que le nettoyage a bien été appliqué.


In [8]:
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids AVANT nettoyage :", buf.getvalue().splitlines()[-1])



Poids AVANT nettoyage : memory usage: 707.2 MB


In [9]:
# Suppression des doublons
df_effectifs = df_effectifs.drop_duplicates()


In [10]:
# Afficher uniquement la ligne "memory usage"
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids APRÈS nettoyage :", buf.getvalue().splitlines()[-1])

Poids APRÈS nettoyage : memory usage: 141.4 MB


In [11]:
df_effectifs.head()


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,libelle_region,departement,libelle_departement
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
5,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,60,NaN,2470,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,60,Oise
10,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,62,NaN,5010,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,62,Pas-de-Calais
15,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,80,NaN,2220,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,80,Somme
20,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,44,10,NaN,1570,NaN,3,plus de 95 ans,tous sexes,78.0,Grand Est,10,Aube


In [12]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

annee                       0
patho_niv1                  0
patho_niv2             101808
patho_niv3             220584
top                         0
cla_age_5                   0
sexe                        0
region                      0
dept                        0
Ntop                   259584
Npop                        0
prev                   259584
Niveau prioritaire      12726
libelle_classe_age          0
libelle_sexe                0
tri                     12726
libelle_region              0
departement                 0
libelle_departement         0
dtype: int64

Dans ce jeu de données, les valeurs indiquées comme NaN ne correspondent pas à de véritables valeurs manquantes. Elles apparaissent simplement lorsqu’un département ne présente aucun cas pour une pathologie donnée. Dans ce contexte, il est donc logique de remplacer ces NaN par 0, afin de refléter correctement l’absence de cas observés.

In [13]:
df_effectifs = df_effectifs.fillna(0)


###  Après avoir passer les NAN à 0

In [14]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

annee                  0
patho_niv1             0
patho_niv2             0
patho_niv3             0
top                    0
cla_age_5              0
sexe                   0
region                 0
dept                   0
Ntop                   0
Npop                   0
prev                   0
Niveau prioritaire     0
libelle_classe_age     0
libelle_sexe           0
tri                    0
libelle_region         0
departement            0
libelle_departement    0
dtype: int64

### Nettoyage et préparation des données

- J’ai décidé de supprimer les parenthèses dans les variables *patho_niv2* et *patho_niv3*, car elles entraînaient un affichage peu lisible dans les visualisations et ajoutaient trop d’informations inutiles.
- Je me suis concentrée sur la France hexagonale, en incluant la Corse et les DOM.
- Les codes **FRANCE** et **Tout département** correspondent à des agrégats régionaux ou nationaux (valeurs non rattachées à un département).  
  Comme ils faussent l’analyse territoriale, j’ai choisi de les supprimer.


In [15]:
# Nettoyage des parenthèses
cols = ["patho_niv2", "patho_niv3"]
for c in cols:
    df_effectifs.loc[:, c] = df_effectifs[c].str.replace(r"\s*\(.*\)", "", regex=True)

# Grand filtre de nettoyage , je veux gardé que la france hexagonale
hors_hexa = ['Tout département', 'Guadeloupe ', 'Haute-Corse', 'Martinique', 'La Réunion', 'Guyane', 'Mayotte', 'Corse-du-Sud','FRANCE']

df_effectifs = df_effectifs[
    (~df_effectifs['libelle_departement'].astype(str).isin(hors_hexa)) &
    
    # On enlève la région "France" qui englobe tout
    (df_effectifs['libelle_region'].astype(str) != 'FRANCE')
]


Pour ne conserver que les observations réellement exploitables, je filtre le dataset
en gardant uniquement les lignes où la variable `prev` est strictement supérieure à 0.
Cela permet d’éliminer les enregistrements sans prévalence et d’alléger les analyses
et visualisations suivantes.

In [16]:
df_effectifs = df_effectifs[df_effectifs["prev"] > 0]


In [17]:
df_effectifs.dtypes


annee                    int64
patho_niv1                 str
patho_niv2              object
patho_niv3              object
top                        str
cla_age_5                  str
sexe                     int64
region                   int64
dept                       str
Ntop                   float64
Npop                     int64
prev                   float64
Niveau prioritaire      object
libelle_classe_age         str
libelle_sexe               str
tri                    float64
libelle_region             str
departement                str
libelle_departement        str
dtype: object

### Nettoyage des colonnes `patho_niv2` et `patho_niv3` après remplacement des `NaN`

Lors du remplacement des valeurs manquantes (`NaN`) par `0`, les colonnes `patho_niv2` et
`patho_niv3` ont changé de type : elles sont passées du type *string* au type *object*.
Lorsqu’une colonne contient un mélange de valeurs c'est ce que pandas fait numériques, textuelles ou des `NaN`, elle est automatiquement convertie en `object`.
Comme les `NaN` avaient été remplacés par `0`, ces valeurs sont restées sous forme de `"0"` après conversion en `str`.  
Je les ai supprimé pour ne conserver que les niveaux de pathologie valides.


In [18]:
df_effectifs["patho_niv2"] = df_effectifs["patho_niv2"].astype(str).str.strip()
df_effectifs["patho_niv3"] = df_effectifs["patho_niv3"].astype(str).str.strip()


In [19]:
df_effectifs.dtypes


annee                    int64
patho_niv1                 str
patho_niv2                 str
patho_niv3                 str
top                        str
cla_age_5                  str
sexe                     int64
region                   int64
dept                       str
Ntop                   float64
Npop                     int64
prev                   float64
Niveau prioritaire      object
libelle_classe_age         str
libelle_sexe               str
tri                    float64
libelle_region             str
departement                str
libelle_departement        str
dtype: object

In [20]:
df_effectifs = df_effectifs[
    (df_effectifs["patho_niv1"] != "0") &
    (df_effectifs["patho_niv2"] != "0") &
    (df_effectifs["patho_niv3"] != "0")
]

### Suppression des colonnes non utilisées

Dans cette étape, je retire les colonnes qui ne sont pas nécessaires pour l’analyse finale.  
Il s’agit principalement de variables techniques, de codes intermédiaires ou de niveaux d’agrégation
qui ne sont pas exploités dans les visualisations (ex. : codes de tri, niveaux prioritaires, variables
de regroupement, ou encore des informations redondantes comme la colonne **region** issue d’un des
datasets utilisés pour la jointure).

L’option `errors='ignore'` permet d’éviter une erreur si certaines colonnes ont déjà été supprimées
lors d’étapes précédentes du nettoyage.


In [21]:
# Suppression des colonnes inutiles pour le nettoyage final
df_effectifs = df_effectifs.drop(
    columns=[
        "Code département",
        "tri",
        "Niveau prioritaire",
        "region",
        "dept",
        "departement",
        "cla_age_5",
        "top",
        "sexe",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




### Renommage de certaines colonnes

Pour améliorer la lisibilité du dataset et faciliter la compréhension des analyses,
j’ai choisi de renommer plusieurs colonnes.  
Afin d’obtenir des intitulés plus courts, plus explicites et cohérents avec les visualisations produites par la suite.



In [22]:
# Renommage des colonens
df_effectifs = df_effectifs.rename(columns={
    "libelle_departement": "Département",
    "libelle_region": "Région",
    "prev" : "Prévalence",
    "Npop": "Population de référence",
    "Ntop": "Effectif",
    "libelle_sexe" : "Sexe",
    "libelle_classe_age" : "Classe d'age"


})

In [23]:
df_effectifs = df_effectifs.dropna()

In [24]:
df_effectifs

,annee,patho_niv1,patho_niv2,patho_niv3,Effectif,Population de référence,Prévalence,Classe d'age,Sexe,Région,Département
150,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,150.0,981490,0.015,tous âges,hommes,Ile-de-France,Paris
155,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,30.0,204050,0.015,tous âges,hommes,Centre-Val de Loire,Eure-et-Loir
160,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,20.0,123680,0.016,tous âges,hommes,Bourgogne-Franche-Comté,Jura
170,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,40.0,259740,0.014,tous âges,hommes,Bourgogne-Franche-Comté,Saône-et-Loire
175,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,10.0,65260,0.018,tous âges,hommes,Bourgogne-Franche-Comté,Territoire de Belfort
...,...,...,...,...,...,...,...,...,...,...,...
4878265,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,47150,0.091,de 45 à 49 ans,femmes,Pays de la Loire,Loire-Atlantique
4878270,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,21550,0.162,de 45 à 49 ans,femmes,Pays de la Loire,Vendée
4878275,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,30.0,17860,0.174,de 45 à 49 ans,femmes,Bretagne,Côtes-d'Armor
4878280,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,34710,0.107,de 45 à 49 ans,femmes,Bretagne,Ille-et-Vilaine


### Exploration des niveaux de pathologies

Avant d’appliquer des filtres plus précis, j’affiche les valeurs uniques de `patho_niv2` et `patho_niv3`
(en excluant les valeurs `NaN`). Cela permet de vérifier la cohérence des libellés et d’identifier
éventuels regroupements ou corrections à effectuer avant l’analyse.


In [25]:
print(df_effectifs['patho_niv1'].dropna().unique())
print(df_effectifs['patho_niv2'].dropna().unique())
print(df_effectifs['patho_niv3'].dropna().unique())


<StringArray>
[                                                               'Maladies inflammatoires ou rares ou infection VIH',
                                                                                           'Maladies neurologiques',
                                                                                          'Maladies psychiatriques',
 'Pas de pathologie repérée, traitement, maternité, hospitalisation ou traitement antalgique ou anti-inflammatoire',
                                                                                   'Total consommants tous régimes',
                                                              'Traitements du risque vasculaire (hors pathologies)',
                                                                      'Traitements psychotropes (hors pathologies)',
                                                                                  'Maladies cardioneurovasculaires',
                                                  

# 2 ème dataset que je vais étudier :

# Dépenses remboursées affectées à chaque pathologie

Les données présentent des informations sur les dépenses remboursées par l’ensemble des régimes d’assurance maladie et affectées à chaque pathologie, traitement chronique ou épisode de soins. Ces dépenses sont réparties en trois grandes catégories :

* les soins de ville (consultations médicales, soins infirmiers, actes de kinésithérapie, médicaments, biologie, transports, etc.)  ; 

* les hospitalisations dans les établissements de santé publics ou privés ;

* les prestations en espèces, notamment les indemnités journalières.

## Deux types d’indicateurs sont proposés pour chacune de ces catégories : 

* les dépenses totales annuelles remboursées, qui mesurent le poids financier global d’une pathologie ;

* les dépenses moyennes annuelles remboursées par patient, qui permettent d’apprécier le coût moyen associé à chaque pathologie.

Ces deux indicateurs offrent ainsi une vision complémentaire des dépenses de santé, en combinant à la fois l’impact global et la charge moyenne par patient.

In [26]:
chunks = []

for chunk in pd.read_csv(
    'https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/depenses.csv',
    sep=';',
    chunksize=100_000,
    low_memory=False
):
    filtered = chunk[
        (chunk['annee'] >= 2022) &
        (chunk['montant'] > 0) 
    ]
    chunks.append(filtered)

df_depenses = pd.concat(chunks, ignore_index=True)
df_depenses


,annee,patho_niv1,patho_niv2,patho_niv3,top,dep_niv_1,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy,Niveau prioritaire,tri,type_somme
0,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,3856836,1714310.0,34840.0,2.0,"1,2,3",16.0,Partiel
1,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Total hospitalisations (tous secteurs) rembour...,308350458,1714310.0,1232630.0,180.0,"1,2,3",16.0,Total
2,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Indemnités journalières maladie et AT/MP rembo...,117367790,1714310.0,125760.0,68.0,"1,2,3",16.0,Partiel
3,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Prestations d'invalidité remboursées,263517389,1714310.0,68580.0,154.0,"1,2,3",16.0,Partiel
4,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Soins de ville,Autres dépenses de soins de ville remboursés,9004352,1714310.0,424990.0,5.0,"1,2,3",16.0,Partiel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4265,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Autres produits de santé remboursés,10878069,335640.0,212200.0,32.0,"2,3",55.0,Partiel
4266,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Soins de généralistes remboursés,11874138,335640.0,300910.0,35.0,"2,3",55.0,Partiel
4267,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Soins dentaires remboursés,6030821,335640.0,118450.0,18.0,"2,3",55.0,Partiel
4268,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Total soins de ville remboursés,199465569,335640.0,335630.0,594.0,"2,3",55.0,Total


# Les champs sont les suivants avant nettoyage:

In [27]:
df_depenses.columns


Index(['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'dep_niv_1',
       'dep_niv_2', 'montant', 'Ntop', 'N_recourant_au_poste', 'montant_moy',
       'Niveau prioritaire', 'tri', 'type_somme'],
      dtype='str')

In [28]:
# Suppression des colonnes inutiles pour le nettoyage final
df_depenses = df_depenses.drop(
    columns=[
        "tri",
        "Niveau prioritaire",
        "type_somme",
        "top",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




# Nettoyage des données

In [29]:
(
    df_depenses
    .isna()
    .sum(axis=0)
)

annee                     0
patho_niv1                0
patho_niv2              457
patho_niv3              997
dep_niv_1                 0
dep_niv_2                 0
montant                   0
Ntop                     28
N_recourant_au_poste     28
montant_moy              28
dtype: int64

In [30]:
df_depenses = df_depenses.drop_duplicates()


In [31]:
df_depenses


,annee,patho_niv1,patho_niv2,patho_niv3,dep_niv_1,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy
0,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,3856836,1714310.0,34840.0,2.0
1,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Hospitalisations (tous secteurs),Total hospitalisations (tous secteurs) rembour...,308350458,1714310.0,1232630.0,180.0
2,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Prestations en espèces,Indemnités journalières maladie et AT/MP rembo...,117367790,1714310.0,125760.0,68.0
3,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Prestations en espèces,Prestations d'invalidité remboursées,263517389,1714310.0,68580.0,154.0
4,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Soins de ville,Autres dépenses de soins de ville remboursés,9004352,1714310.0,424990.0,5.0
...,...,...,...,...,...,...,...,...,...,...
4265,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Autres produits de santé remboursés,10878069,335640.0,212200.0,32.0
4266,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Soins de généralistes remboursés,11874138,335640.0,300910.0,35.0
4267,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Soins dentaires remboursés,6030821,335640.0,118450.0,18.0
4268,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Total soins de ville remboursés,199465569,335640.0,335630.0,594.0


# Filtrage avant creation du dashboard

In [32]:
df_depenses = df_depenses.drop('Total consommants tous régimes', errors='ignore')

# Création du fichier Excel

Je vais créer un nouveau fichier Excel qui contiendra les données nettoyées dans un onglet "cleanedData_depenses" et "cleanedData_Pathos"

Pour les dépenses et pour les pathologies :

In [33]:
output_dep = '../DATA/Dashboard_Depenses.xlsx'

with pd.ExcelWriter(output_dep, engine='openpyxl') as writer:
    df_depenses.to_excel(writer, sheet_name='cleanedData_depenses', index=False)

print(f"✓ Fichier dépenses créé : {output_dep}")


✓ Fichier dépenses créé : ../DATA/Dashboard_Depenses.xlsx


In [34]:
output_eff = '../DATA/Dashboard_Effectifs.xlsx'

with pd.ExcelWriter(output_eff, engine='openpyxl') as writer:
    df_effectifs.to_excel(writer, sheet_name='cleanedData_Effectifs', index=False)

print(f"✓ Fichier effectifs créé : {output_eff}")


✓ Fichier effectifs créé : ../DATA/Dashboard_Effectifs.xlsx


In [35]:
wb = load_workbook(output_path)


NameError: name 'output_path' is not defined

Calcul des listes _ len_dict

In [ ]:
annees = sorted(df_depenses['annee'].dropna().unique())
pathos = sorted(df_depenses['patho_niv1'].dropna().unique())
postes = sorted(df_depenses['dep_niv_1'].dropna().unique())

# LISTES POUR FILTRES EFFECTIFS
depts = sorted(df_effectifs['Département'].dropna().unique())
classes_age = sorted(df_effectifs['Classe d\'age'].dropna().unique())
sexes = sorted(df_effectifs['Sexe'].dropna().unique())
regions = sorted(df_effectifs['Région'].dropna().unique())
pathos_eff = sorted(df_effectifs['patho_niv1'].dropna().unique())

# LEN_DICT
len_dict = {
    'len_annee': len(annees) + 1,
    'len_patho_niv1': len(pathos) + 1,
    'len_dep_niv_1': len(postes) + 1,
    'len_departement_eff': len(depts) + 1,
    'len_classes_age': len(classes_age) + 1,
    'len_sexe': len(sexes) + 1,
    'len_region': len(regions) + 1,
    'len_patho_eff': len(pathos_eff) + 1, 
}

import pprint
pprint.pprint(len_dict)

{'len_annee': 3,
 'len_classes_age': 22,
 'len_dep_niv_1': 5,
 'len_departement_eff': 96,
 'len_patho_eff': 19,
 'len_patho_niv1': 20,
 'len_region': 14,
 'len_sexe': 4}


In [ ]:
cols_to_calculate = ['annee', 'patho_niv1', 'dep_niv_1', 'dep_niv_2']

cols_to_calculate_effectifs = {
    'annee_eff': 'annee',
    'patho_effectif': 'patho_niv1',
    'age': "Classe d'age",
    'sexe': 'Sexe',
    'region': 'Région',
    'departement_eff': 'Département'
}
len_dict = {}

# BOUCLE POUR DÉPENSES
for col in cols_to_calculate:
    if col == 'patho_niv1':
        # Exclure "Total"
        uniques = df_depenses.loc[
            ~df_depenses[col].isin(['Total', 'Total consommants tous régimes']),
            col
        ].dropna().unique()
    else:
        uniques = df_depenses[col].dropna().unique()
    
    len_dict[f"len_{col}"] = len(uniques) + 1

# BOUCLE POUR EFFECTIFS
for key_name, col_real_name in cols_to_calculate_effectifs.items():
    if col_real_name in df_effectifs.columns:
        if key_name == 'age':
            # Exclure 'tous âges', 'tous ages', 'total'
            uniques = df_effectifs.loc[
                ~df_effectifs[col_real_name].astype(str).str.lower().isin(['tous âges', 'tous ages', 'total']),
                col_real_name
            ].dropna().unique()
        elif key_name == 'patho_effectif':
            # Exclure 'total'
            uniques = df_effectifs.loc[
                ~df_effectifs[col_real_name].astype(str).str.lower().isin(['total']),
                col_real_name
            ].dropna().unique()
        else:
            uniques = df_effectifs[col_real_name].dropna().unique()
        
        len_dict[f"len_{key_name}"] = len(uniques) + 1

# Dépenses
annees = sorted(df_depenses['annee'].dropna().unique())

pathos = sorted(df_depenses.loc[
    ~df_depenses['patho_niv1'].isin(['Total', 'Total consommants tous régimes']),
    'patho_niv1'
].dropna().unique())

postes = sorted(df_depenses['dep_niv_1'].dropna().unique())
postes_niv2 = sorted(df_depenses['dep_niv_2'].dropna().unique())

# Effectifs
depts = sorted(df_effectifs['Département'].dropna().unique())

classes_age = sorted(df_effectifs.loc[
    ~df_effectifs['Classe d\'age'].astype(str).str.lower().isin(['tous âges', 'tous ages', 'total']),
    'Classe d\'age'
].dropna().unique())

sexes = sorted(df_effectifs['Sexe'].dropna().unique())
regions = sorted(df_effectifs['Région'].dropna().unique())

pathos_eff = sorted(df_effectifs.loc[
    ~df_effectifs['patho_niv1'].astype(str).str.lower().isin(['Total', 'Total consommants tous régimes']),
    'patho_niv1'
].dropna().unique())

pprint.pprint(len_dict)

{'len_age': 21,
 'len_annee': 3,
 'len_annee_eff': 3,
 'len_dep_niv_1': 5,
 'len_dep_niv_2': 32,
 'len_departement_eff': 96,
 'len_patho_effectif': 19,
 'len_patho_niv1': 19,
 'len_region': 14,
 'len_sexe': 4}


In [ ]:
# === COULEURS ===
COULEURS = {
    'bleu_fonce': 'FF1F4E78',
    'bleu_clair': 'FF4472C4',
    'bleu_fonce': 'FF00CCFF',
    'jaune_filtre': 'FFF2CC',
    'gris_clair': 'FFCCCCCC',
}

# Noms des sheets
sheet_dep = 'cleanedData_depenses'
sheet_eff = 'cleanedData_Effectifs'

# Charger le workbook existant
wb = load_workbook(output_path)
wb.save(output_path)
wb.close()


Création d'un onglet Filtres

Cet onglet est nécessaire pour alimenter les listes de validaiton dans les filtres.

Je vais récupérer l'année, la pathologie et le poste de dépense avec la fonction UNIQUE() depuis les onglets "cleaned_data"

In [ ]:
from openpyxl.utils import FORMULAE
"UNIQUE" in FORMULAE

False

In [ ]:
wb = load_workbook(output_path)

if 'Filtres' not in wb.sheetnames:
    filtres_sheet = wb.create_sheet('Filtres', 0)
else:
    filtres_sheet = wb['Filtres']
    filtres_sheet.delete_rows(1, filtres_sheet.max_row)

formula_annee = f"=_xlfn.SORT(_xlfn.UNIQUE({sheet_dep}!A:A))"
filtres_sheet['A1'] = ArrayFormula(f"A1:A{len_dict['len_annee']}", formula_annee)

formula_patho = f"=_xlfn.UNIQUE({sheet_dep}!B:B)"
filtres_sheet['B1'] = ArrayFormula(f"B1:B{len_dict['len_patho_niv1']}", formula_patho)

formula_poste = f"=_xlfn.UNIQUE({sheet_dep}!E:E)"
filtres_sheet['C1'] = ArrayFormula(f"C1:C{len_dict['len_dep_niv_1']}", formula_poste)

formula_poste_niv2 = f"=_xlfn.UNIQUE({sheet_dep}!F:F)"
filtres_sheet['D1'] = ArrayFormula(f"D1:D{len_dict['len_dep_niv_2']}", formula_poste_niv2)

formula_annee_eff = f"=_xlfn.UNIQUE({sheet_eff}!A:A)"
filtres_sheet['E1'] = ArrayFormula(f"E1:E{len_dict['len_annee_eff']}", formula_annee_eff)

formula_patho_eff = f"=_xlfn.UNIQUE({sheet_eff}!B:B)"
filtres_sheet['F1'] = ArrayFormula(f"F1:F{len_dict['len_patho_effectif']}", formula_patho_eff)

formula_age = f"=_xlfn.UNIQUE({sheet_eff}!H:H)"
filtres_sheet['G1'] = ArrayFormula(f"G1:G{len_dict['len_age']}", formula_age)

formula_sexe = f"=_xlfn.UNIQUE({sheet_eff}!I:I)"
filtres_sheet['H1'] = ArrayFormula(f"H1:H{len_dict['len_sexe']}", formula_sexe)

formula_region = f"=_xlfn.UNIQUE({sheet_eff}!J:J)"
filtres_sheet['I1'] = ArrayFormula(f"I1:I{len_dict['len_region']}", formula_region)

formula_dept = f"=_xlfn.UNIQUE({sheet_eff}!K:K)"
filtres_sheet['J1'] = ArrayFormula(f"J1:J{len_dict['len_departement_eff']}", formula_dept)
wb.save(output_path)
wb.close()


## Fonctions utilitaires

Comme nous allons répéter cette opération plusieurs fois, il est préférable de créer une fonction qui effectue les mêmes opérations.

In [ ]:
wb = load_workbook(output_path)

def auto_adjust_columns(sheet: Worksheet) -> None:
    """Ajuste automatiquement la largeur des colonnes selon le contenu."""
    for col_idx, column in enumerate(sheet.columns, 1):
        max_length = 0
        col_letter = get_column_letter(col_idx)
        for cell in column:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        adjusted_width = min(max_length + 2, 50)
        sheet.column_dimensions[col_letter].width = adjusted_width

def add_filter_title(sheet: Worksheet, title: str, start_row: int, start_column: int) -> None:
    """Ajoute le titre du filtre."""
    cell = sheet.cell(row=start_row, column=start_column)
    cell.value = title
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.fill = PatternFill(start_color='FFD3D3D3', fill_type='solid')
    cell.font = Font(bold=True)

def add_filter_value(sheet: Worksheet, value, start_row: int, start_column: int) -> None:
    """Ajoute la valeur du filtre."""
    cell = sheet.cell(row=start_row, column=start_column)
    cell.value = value
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.fill = PatternFill(start_color='FFE7E6E6', fill_type='solid')
    cell.font = Font(bold=True)

def add_data_validation(sheet: Worksheet, start_row: int, start_column: int, formula: str) -> None:
    """Ajoute la validation de données."""
    dv = DataValidation(type='list', formula1=formula)
    sheet.add_data_validation(dv)
    dv.add(sheet.cell(row=start_row, column=start_column).coordinate)

def add_filter(sheet: Worksheet, title: str, value, start_row: int, start_column: int, end_row: int, end_column: int, formula: str) -> None:
    """Ajoute un filtre complet."""
    target_val_column = end_column + 1
    add_filter_title(sheet, title, start_row, start_column)
    add_filter_value(sheet, value, start_row, target_val_column)
    add_data_validation(sheet, start_row, target_val_column, formula)
    sheet.merge_cells(start_row=start_row, start_column=start_column, end_row=end_row, end_column=end_column)
    sheet.merge_cells(start_row=start_row, start_column=target_val_column, end_row=end_row, end_column=target_val_column + 1)
wb.save(output_path)
wb.close()

Création du dashboard

Je vais créer un premier tableau de bord interactif avec des filtres.
Je vais me concentrer sur les dépenses de remboursement par pathologie et par poste de dépense.
L'utilisateur aura la possibilité de choisir l'année, la pathologie et le poste de dépense via des filtres sous forme de listes de validation.
Ces filtres permettront d'actualiser dynamiquement les formules et les graphiques.

Design :

In [ ]:
wb = load_workbook(output_path)

COULEURS = {
    'bleu_fonce': 'FF1F4E78',
    'bleu_clair': 'FF4F81BD',
    'jaune_filtre': 'FFF2CC',
    'cyan': 'FF00BCD4',
    'gris_clair': 'CCCCCC'
}
trait = Side(style='thin', color=COULEURS['gris_clair'])
bordure = Border(left=trait, right=trait, top=trait, bottom=trait)

wb.save(output_path)
wb.close()

## Onglet postes

In [ ]:
wb = load_workbook(output_path)

if 'Postes' not in wb.sheetnames:
    postes_sheet = wb.create_sheet('Postes', 1)
else:
    postes_sheet = wb['Postes']
    postes_sheet.delete_rows(1, postes_sheet.max_row)

postes_sheet.sheet_view.showGridLines = False

# Filtre Année
add_filter(
    sheet=postes_sheet, title='Année', value=int(annees[-1]),
    start_row=1, start_column=1, end_row=2, end_column=2,
    formula=f"=Filtres!$A$2:$A${len_dict['len_annee']}"
)

# Titre
postes_sheet['A3'] = "Analyse par postes de dépenses"
postes_sheet['A3'].font = Font(bold=True, size=14, color=COULEURS['bleu_fonce'])
postes_sheet['A3'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
postes_sheet.merge_cells('A3:E3')

# KPI
postes_sheet['A5'] = "Total dépenses"
postes_sheet['A5'].font = Font(bold=True)
postes_sheet['A6'] = f"=IFERROR(SUMIFS('{sheet_dep}'!G:G,'{sheet_dep}'!A:A,C$1),0)"
postes_sheet['A6'].font = Font(bold=True, size=14, color=COULEURS['bleu_fonce'])
postes_sheet['A6'].number_format = '#,##0'

# Headers (ligne 8)
headers = ['Poste', 'Dépenses', 'Patients', 'Coût/patient']
for col_idx, header in enumerate(headers, 1):
    cell = postes_sheet.cell(row=8, column=col_idx)
    cell.value = header
    cell.font = Font(bold=True, color='FFFFFF')
    cell.fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')
    cell.alignment = Alignment(horizontal='center')

# Données (à partir de la ligne 9)
row = 9
for poste in postes:
    postes_sheet[f'A{row}'] = poste
    postes_sheet[f'B{row}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!G:G,'{sheet_dep}'!A:A,C$1,'{sheet_dep}'!E:E,A{row}),0)"
    postes_sheet[f'C{row}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!H:H,'{sheet_dep}'!A:A,C$1,'{sheet_dep}'!E:E,A{row}),0)"
    postes_sheet[f'D{row}'] = f"=IFERROR(B{row}/C{row},0)"
    for col in ['B', 'C', 'D']:
        postes_sheet[f'{col}{row}'].number_format = '#,##0'
    row += 1

# Ajustement et largeurs
auto_adjust_columns(postes_sheet)
postes_sheet.column_dimensions['A'].width = 38
postes_sheet.column_dimensions['B'].width = 22

# Graphique
chart = BarChart()
chart.type = "bar"
chart.title = "Distribution par poste"
chart.height = 12
chart.width = 18
chart.add_data(Reference(postes_sheet, min_col=2, min_row=9, max_row=row-1), titles_from_data=False)
chart.set_categories(Reference(postes_sheet, min_col=1, min_row=9, max_row=row-1))
if chart.series:
    chart.series[0].title = SeriesLabel(v="Dépenses")
postes_sheet.add_chart(chart, "G5")

wb.save(output_path)
wb.close()

## Onglet Pathologies

In [ ]:
wb = load_workbook(output_path)

if 'Pathologies' not in wb.sheetnames:
    pathos_sheet = wb.create_sheet('Pathologies', 2)
else:
    pathos_sheet = wb['Pathologies']
    pathos_sheet.delete_rows(1, pathos_sheet.max_row)

pathos_sheet.sheet_view.showGridLines = False

# Filtres
add_filter(
    sheet=pathos_sheet, title='Année', value=int(annees[-1]),
    start_row=1, start_column=1, end_row=2, end_column=2,
    formula=f"=Filtres!$A$2:$A${len_dict['len_annee']}"
)

add_filter(
    sheet=pathos_sheet, title='Grand Poste', value=postes[0] if postes else 'N/A',
    start_row=1, start_column=5, end_row=2, end_column=6,
    formula=f"=Filtres!$C$2:$C${len_dict['len_dep_niv_1']}"
)

# Titre
pathos_sheet['A3'] = "Analyse par pathologies"
pathos_sheet['A3'].font = Font(bold=True, size=14, color=COULEURS['cyan'])
pathos_sheet['A3'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
pathos_sheet.merge_cells('A3:E3')

# KPI
pathos_sheet['A5'] = "Total dépenses"
pathos_sheet['A5'].font = Font(bold=True)

# Headers (ligne 8)
headers_patho = ['Pathologie', 'Dépenses', 'Patients', 'Coût/patient']
for col_idx, header in enumerate(headers_patho, 1):
    cell = pathos_sheet.cell(row=8, column=col_idx)
    cell.value = header
    cell.font = Font(bold=True, color='FFFFFF')
    cell.fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')
    cell.alignment = Alignment(horizontal='center')

# Données (à partir de la ligne 9)
row_patho = 9
for patho in pathos:
    pathos_sheet[f'A{row_patho}'] = patho
    pathos_sheet[f'A{row_patho}'].font = Font(size=10)
    pathos_sheet[f'B{row_patho}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!G:G, '{sheet_dep}'!A:A, C$1, '{sheet_dep}'!E:E, F$1, '{sheet_dep}'!B:B, A{row_patho}), 0)"
    pathos_sheet[f'C{row_patho}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!H:H, '{sheet_dep}'!A:A, C$1, '{sheet_dep}'!E:E, F$1, '{sheet_dep}'!B:B, A{row_patho}), 0)"
    pathos_sheet[f'D{row_patho}'] = f"=IFERROR(B{row_patho}/C{row_patho}, 0)"
    for col in ['B', 'C', 'D']:
        pathos_sheet[f'{col}{row_patho}'].number_format = '#,##0'
        pathos_sheet[f'{col}{row_patho}'].font = Font(size=10)
    row_patho += 1

# KPI total
pathos_sheet['A6'] = f"=IFERROR(SUM(B9:B{row_patho-1}), 0)"
pathos_sheet['A6'].font = Font(bold=True, size=14, color=COULEURS['bleu_fonce'])
pathos_sheet['A6'].number_format = '#,##0'

# Ajustement
auto_adjust_columns(pathos_sheet)
pathos_sheet.column_dimensions['A'].width = 32
pathos_sheet.column_dimensions['B'].width = 22

# Graphique Pie
chart2 = PieChart()
chart2.title = "Distribution par pathologie"
chart2.height = 12
chart2.width = 18
if row_patho > 9:
    chart2.add_data(Reference(pathos_sheet, min_col=2, min_row=9, max_row=row_patho-1), titles_from_data=False)
    chart2.set_categories(Reference(pathos_sheet, min_col=1, min_row=9, max_row=row_patho-1))
    if chart2.series:
        chart2.series[0].title = SeriesLabel(v="Dépenses")
    pathos_sheet.add_chart(chart2, "H5")

wb.save(output_path)
wb.close()

## Onglet Effectifs

In [ ]:
wb = load_workbook(output_path)

if 'Effectifs' not in wb.sheetnames:
    effectifs_sheet = wb.create_sheet('Effectifs', 3)
else:
    effectifs_sheet = wb['Effectifs']
    effectifs_sheet.delete_rows(1, effectifs_sheet.max_row)

effectifs_sheet.sheet_view.showGridLines = False

# TITRE
effectifs_sheet['A1'] = "Diagnostic national des pathologies"
effectifs_sheet['A1'].font = Font(bold=True, size=12, color='FFFFFF')
effectifs_sheet['A1'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
effectifs_sheet.merge_cells('A1:M1')

# FILTRES
effectifs_sheet['A3'] = 'Département'
effectifs_sheet['A3'].font = Font(bold=True, color='FFFFFF')
effectifs_sheet['A3'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
effectifs_sheet['B3'] = depts[0] if depts else "N/A"
effectifs_sheet['B3'].font = Font(bold=True)
effectifs_sheet['B3'].fill = PatternFill(start_color=COULEURS['jaune_filtre'], fill_type='solid')
dv_dept = DataValidation(type='list', formula1=f"=Filtres!$D$2:$D${len_dict['len_departement_eff']}")
effectifs_sheet.add_data_validation(dv_dept)
dv_dept.add('B3')

effectifs_sheet['A4'] = 'Année'
effectifs_sheet['A4'].font = Font(bold=True, color='FFFFFF')
effectifs_sheet['A4'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
effectifs_sheet['B4'] = int(annees[-1])
effectifs_sheet['B4'].font = Font(bold=True)
effectifs_sheet['B4'].fill = PatternFill(start_color=COULEURS['jaune_filtre'], fill_type='solid')
dv_annee = DataValidation(type='list', formula1=f"=Filtres!$A$2:$A${len_dict['len_annee']}")
effectifs_sheet.add_data_validation(dv_annee)
dv_annee.add('B4')

# === TABLE 1 : Classe d'âge ===
effectifs_sheet['A6'] = "Classe d'âge"
effectifs_sheet['B6'] = "Effectif"
for col in ['A', 'B']:
    effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row = 7
for classe_age in classes_age:
    effectifs_sheet[f'A{row}'] = classe_age
    effectifs_sheet[f'B{row}'] = f"=IFERROR(SUMIFS('{sheet_eff}'!E:E,'{sheet_eff}'!A:A,B$4,'{sheet_eff}'!K:K,B$3,'{sheet_eff}'!H:H,A{row}),0)"
    effectifs_sheet[f'B{row}'].number_format = '#,##0'
    row += 1

# === TABLE 2 : TOP 10 Classe d'âge ===
top_class = df_effectifs.groupby('Classe d\'age')['Effectif'].sum().sort_values(ascending=False).head(10)

effectifs_sheet['D6'] = "Classe d'âge (TOP)"
effectifs_sheet['E6'] = "Effectif"
for col in ['D', 'E']:
    effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row_top = 7
for classe, effectif in top_class.items():
    effectifs_sheet[f'D{row_top}'] = classe
    effectifs_sheet[f'E{row_top}'] = effectif
    effectifs_sheet[f'E{row_top}'].number_format = '#,##0'
    row_top += 1

# === TABLE 3 : Sexe ===
top_sex = df_effectifs.groupby('Sexe')['Effectif'].sum().sort_values(ascending=False)

effectifs_sheet['G6'] = "Sexe"
effectifs_sheet['H6'] = "Effectif"
for col in ['G', 'H']:
    effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row_sex = 7
for sexe, effectif in top_sex.items():
    effectifs_sheet[f'G{row_sex}'] = sexe
    effectifs_sheet[f'H{row_sex}'] = effectif
    effectifs_sheet[f'H{row_sex}'].number_format = '#,##0'
    row_sex += 1

# TOP 10 Département
top_dept = df_effectifs.groupby('Département')['Effectif'].sum().sort_values(ascending=False).head(10)

effectifs_sheet['J6'] = "Département"
effectifs_sheet['K6'] = "Effectif"
for col in ['J', 'K']:
    effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row_dept = 7
for dept, effectif in top_dept.items():
    effectifs_sheet[f'J{row_dept}'] = dept
    effectifs_sheet[f'K{row_dept}'] = effectif
    effectifs_sheet[f'K{row_dept}'].number_format = '#,##0'
    row_dept += 1

wb.save(output_path)
wb.close()

In [ ]:
wb = load_workbook(output_path)

if 'Depenses' in wb.sheetnames:
    del wb['Depenses']
depenses_sheet = wb.create_sheet('Depenses', 4)

depenses_sheet.sheet_view.showGridLines = False

depenses_sheet['A1'] = "Analyse détaillée des dépenses par pathologie"
depenses_sheet['A1'].font = Font(bold=True, size=12, color='FFFFFF')
depenses_sheet['A1'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
depenses_sheet.merge_cells('A1:M1')

depenses_sheet['A3'] = 'Dépense Niv1'
depenses_sheet['A3'].font = Font(bold=True, color='FFFFFF')
depenses_sheet['A3'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
depenses_sheet['B3'] = postes[0] if postes else "N/A"
depenses_sheet['B3'].font = Font(bold=True)
depenses_sheet['B3'].fill = PatternFill(start_color=COULEURS['jaune_filtre'], fill_type='solid')
dv_poste = DataValidation(type='list', formula1=f"=Filtres!$C$2:$C${len_dict['len_dep_niv_1']}")
depenses_sheet.add_data_validation(dv_poste)
dv_poste.add('B3')

depenses_sheet['A4'] = 'Année'
depenses_sheet['A4'].font = Font(bold=True, color='FFFFFF')
depenses_sheet['A4'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
depenses_sheet['B4'] = int(annees[-1])
depenses_sheet['B4'].font = Font(bold=True)
depenses_sheet['B4'].fill = PatternFill(start_color=COULEURS['jaune_filtre'], fill_type='solid')
dv_annee_dep = DataValidation(type='list', formula1=f"=Filtres!$A$2:$A${len_dict['len_annee']}")
depenses_sheet.add_data_validation(dv_annee_dep)
dv_annee_dep.add('B4')

depenses_sheet['A6'] = "Pathologie"
depenses_sheet['B6'] = "Dépenses (€)"
depenses_sheet['C6'] = "Coût par patient (€)"

for col in ['A', 'B', 'C']:
    depenses_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
    depenses_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row_idx = 7
liste_pathologies = pathos_eff if 'pathos_eff' in locals() else (df_effectifs['Pathologie'].unique() if 'df_effectifs' in locals() else [])

for patho in liste_pathologies:
    depenses_sheet[f'A{row_idx}'] = patho
    depenses_sheet[f'B{row_idx}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!E:E, '{sheet_dep}'!A:A, B$4, '{sheet_dep}'!C:C, B$3, '{sheet_dep}'!G:G, A{row_idx}), 0)"
    depenses_sheet[f'B{row_idx}'].number_format = '#,##0'
    
    formula_effectif = f"SUMIFS('{sheet_eff}'!E:E, '{sheet_eff}'!A:A, B$4, '{sheet_eff}'!G:G, A{row_idx})"
    depenses_sheet[f'C{row_idx}'] = f"=IFERROR(B{row_idx} / ({formula_effectif}), 0)"
    depenses_sheet[f'C{row_idx}'].number_format = '#,##0.00'
    
    row_idx += 1

chart_pie = PieChart()
chart_pie.title = "Répartition des Dépenses par Pathologie"
chart_pie.height = 14
chart_pie.width = 18

data_pie = Reference(depenses_sheet, min_col=2, min_row=6, max_row=row_idx-1)
labels_pie = Reference(depenses_sheet, min_col=1, min_row=7, max_row=row_idx-1)

chart_pie.add_data(data_pie, titles_from_data=True)
chart_pie.set_categories(labels_pie)

depenses_sheet.add_chart(chart_pie, "E6")

wb.save(output_path)
wb.close()

Les graphiques :

In [ ]:
wb = load_workbook(output_path)

# === TABLE 1 : Classe d'âge ===
effectifs_sheet['A6'] = "Classe d'âge"
effectifs_sheet['B6'] = "Effectif"
for col in ['A', 'B']:
    effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row = 7
for classe_age in classes_age:
    effectifs_sheet[f'A{row}'] = classe_age
    effectifs_sheet[f'B{row}'] = f"=IFERROR(SUMIFS('{sheet_eff}'!E:E,'{sheet_eff}'!A:A,B$4,'{sheet_eff}'!K:K,B$3,'{sheet_eff}'!H:H,A{row}),0)"
    effectifs_sheet[f'B{row}'].number_format = '#,##0'
    row += 1

# === TABLE 2 : TOP 10 Classe d'âge ===
top_class = df_effectifs.groupby("Classe d'age")['Effectif'].sum().sort_values(ascending=False).head(10)

effectifs_sheet['D6'] = "Classe d'âge (TOP)"
effectifs_sheet['E6'] = "Effectif"
for col in ['D', 'E']:
    effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row_top = 7
for classe, effectif in top_class.items():
    effectifs_sheet[f'D{row_top}'] = classe
    effectifs_sheet[f'E{row_top}'] = effectif
    effectifs_sheet[f'E{row_top}'].number_format = '#,##0'
    row_top += 1

# === TABLE 3 : Sexe ===
top_sex = df_effectifs.groupby('Sexe')['Effectif'].sum().sort_values(ascending=False)

effectifs_sheet['G6'] = "Sexe"
effectifs_sheet['H6'] = "Effectif"
for col in ['G', 'H']:
    effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row_sex = 7
for sexe, effectif in top_sex.items():
    effectifs_sheet[f'G{row_sex}'] = sexe
    effectifs_sheet[f'H{row_sex}'] = effectif
    effectifs_sheet[f'H{row_sex}'].number_format = '#,##0'
    row_sex += 1

# TOP 10 Département
top_dept = df_effectifs.groupby('Département')['Effectif'].sum().sort_values(ascending=False).head(10)

effectifs_sheet['J6'] = "Département"
effectifs_sheet['K6'] = "Effectif"
for col in ['J', 'K']:
    effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row_dept = 7
for dept, effectif in top_dept.items():
    effectifs_sheet[f'J{row_dept}'] = dept
    effectifs_sheet[f'K{row_dept}'] = effectif
    effectifs_sheet[f'K{row_dept}'].number_format = '#,##0'
    row_dept += 1

# === GRAPHIQUE 1 : Histogramme en barres pour le TOP 10 Département ===
chart_bar = BarChart()
chart_bar.type = "col"
chart_bar.style = 10
chart_bar.title = "TOP 10 Effectifs par Département"
chart_bar.y_axis.title = "Effectifs"
chart_bar.x_axis.title = "Départements"

data_dept = Reference(effectifs_sheet, min_col=11, min_row=6, max_row=row_dept-1)
cats_dept = Reference(effectifs_sheet, min_col=10, min_row=7, max_row=row_dept-1)
chart_bar.add_data(data_dept, titles_from_data=True)
chart_bar.set_categories(cats_dept)
chart_bar.legend = None
chart_bar.height = 14
chart_bar.width = 22
effectifs_sheet.add_chart(chart_bar, "A22")

# === GRAPHIQUE 2 : Répartition Pathologies par Sexe (Barres empilées 100%) ===
df_pivot = df_effectifs.groupby(['patho_niv1', 'Sexe'])['Effectif'].sum().unstack().fillna(0)
df_pivot['Total'] = df_pivot.sum(axis=1)
df_pivot = df_pivot.sort_values(by='Total', ascending=False).head(10).drop(columns=['Total'])

effectifs_sheet['O6'] = "Pathologie"
effectifs_sheet['P6'] = "Homme"
effectifs_sheet['Q6'] = "Femme"
for col in ['O', 'P', 'Q']:
    effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row_pivot = 7
for patho, rows in df_pivot.iterrows():
    effectifs_sheet[f'O{row_pivot}'] = patho
    effectifs_sheet[f'P{row_pivot}'] = rows.get('Homme', rows.get('H', rows.get('M', rows.iloc[0])))
    effectifs_sheet[f'Q{row_pivot}'] = rows.get('Femme', rows.get('F', rows.iloc[1] if len(rows) > 1 else 0))
    effectifs_sheet[f'P{row_pivot}'].number_format = '#,##0'
    effectifs_sheet[f'Q{row_pivot}'].number_format = '#,##0'
    row_pivot += 1

chart_stack = BarChart()
chart_stack.type = "bar"
chart_stack.grouping = "percentStacked"
chart_stack.overlap = 100
chart_stack.title = "Répartition Top 10 Pathologies par Sexe (100%)"

data_stack = Reference(effectifs_sheet, min_col=16, max_col=17, min_row=6, max_row=row_pivot-1)
cats_stack = Reference(effectifs_sheet, min_col=15, min_row=7, max_row=row_pivot-1)
chart_stack.add_data(data_stack, titles_from_data=True)
chart_stack.set_categories(cats_stack)
chart_stack.height = 14
chart_stack.width = 22
effectifs_sheet.add_chart(chart_stack, "J22")

wb.save(output_path)
wb.close()

## Sauvegarde finale

In [ ]:
wb.save(output_path)
wb.close()

